# Video to frames converter

In [ ]:
import cv2
import os

# Define input video path and output folder
video_path = '/content/drive/MyDrive/input_video.mp4'
output_folder = '/content/drive/MyDrive/input_frames'

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Open the video file
video = cv2.VideoCapture(video_path)

if not video.isOpened():
    print(f"Error: Could not open video file {video_path}")
else:
    frame_count = 0
    while True:
        ret, frame = video.read()  # Read a frame
        if not ret:
            break  # Exit when no frames are left

        # Save frame as an image
        frame_filename = os.path.join(output_folder, f'frame_{frame_count:04d}.jpg')
        cv2.imwrite(frame_filename, frame)
        frame_count += 1

    print(f"Video converted to frames successfully! Total frames: {frame_count}")
    print(f"Frames are saved in: {output_folder}")

# Release the video object
video.release()


# **Code**

In [ ]:
!pip install ultralytics

In [ ]:
import cv2
import os
import numpy as np
from ultralytics import YOLO
from scipy.spatial import distance as dist

# ————————— PARAMETERS —————————
FRAME_DIR    = "/content/drive/MyDrive/input_frames" # input folder of frames of video
OUTPUT_DIR   = "/content/drive/MyDrive/output_frames_folder" # output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_FRAMES_THRESHOLD = 5
SKIP_DRAW_THRESHOLD  = 2

# YOLOv8 person detector
person_model = YOLO("yolov8l.pt")

# Polygons in image space
poly2 = np.array([[677,358],[607,578],[906,567],[854,352]], dtype=np.int32)  # bike lane
poly1 = np.array([[854,352],[906,567],[1497,553],[1250,338]], dtype=np.int32) # non-bike lane

# Road corners a–f for homography
a, b, c = (677,358), (607,578), (854,352)
d, e, f = (906,567), (1250,338), (1497,553)

# Pixels-per-foot and graph sizes
PPF = 20
W2  = int(12 * PPF)  # 12 ft bike-lane
W1  = int(24 * PPF)  # 24 ft non-bike-lane
# Estimate H from vertical real-world span (12ft) scaled by PPF:
H   = int(12 * PPF)

# Homography src/dst quads
src2 = np.array([a,b,d,c], dtype=np.float32)
src1 = np.array([c,d,f,e], dtype=np.float32)
dst2 = np.array([[0,0],[0,H],[W2,H],[W2,0]], dtype=np.float32)
dst1 = np.array([[W2,0],[W2,H],[W2+W1,H],[W2+W1,0]], dtype=np.float32)

H2 = cv2.getPerspectiveTransform(src2, dst2)
H1 = cv2.getPerspectiveTransform(src1, dst1)

# Create bird’s-eye canvas
canvas = np.full((H, W2+W1, 3), 255, np.uint8)
cv2.line(canvas, (0,0),       (0,H),        (0,0,0), 2)
cv2.line(canvas, (W2,0),      (W2,H),       (0,0,0), 2)
cv2.line(canvas, (W2+W1,0),   (W2+W1,H),    (0,0,0), 2)

# ————————— SIMPLE CENTROID TRACKER —————————
class KalmanCentroidTracker:
    def __init__(self, maxDisappeared=50, distance_threshold=50):
        self.nextID      = 0
        self.objects     = {}
        self.disappeared = {}
        self.framesSeen  = {}
        self.maxDis      = maxDisappeared
        self.distThresh  = distance_threshold

    def register(self, pt):
        self.objects[self.nextID]     = tuple(pt)
        self.disappeared[self.nextID] = 0
        self.framesSeen[self.nextID]  = 1
        self.nextID += 1

    def deregister(self, i):
        for d in (self.objects, self.disappeared, self.framesSeen):
            d.pop(i, None)

    def update(self, pts):
        # no detections
        if not pts:
            toRemove = []
            for i in self.objects:
                self.disappeared[i] += 1
                if self.disappeared[i] > self.maxDis:
                    toRemove.append(i)
            for i in toRemove:
                self.deregister(i)
            return self.objects

        # first time
        if not self.objects:
            for p in pts:
                self.register(p)
            return self.objects

        # build distance matrix
        ids = list(self.objects)
        old = np.array([self.objects[i] for i in ids])
        D   = dist.cdist(old, np.array(pts))
        rows = D.min(axis=1).argsort()
        usedCols = set()

        # match existing
        for r in rows:
            c = D[r].argmin()
            if c in usedCols or D[r,c] > self.distThresh:
                continue
            i = ids[r]
            self.objects[i]     = tuple(pts[c])
            self.disappeared[i] = 0
            self.framesSeen[i] += 1
            usedCols.add(c)

        # new registrations
        for c in set(range(len(pts))) - usedCols:
            self.register(pts[c])

        # disappeared
        usedRows = set(rows)
        for r in set(range(len(ids))) - usedRows:
            i = ids[r]
            self.disappeared[i] += 1
            if self.disappeared[i] > self.maxDis:
                self.deregister(i)

        return self.objects

tracker2 = KalmanCentroidTracker()
tracker1 = KalmanCentroidTracker()
counted2 = set()
counted1 = set()

# ————————— VIDEO WRITERS —————————
fps = 20
# grab first frame to get dimensions
first = cv2.imread(os.path.join(FRAME_DIR, sorted(os.listdir(FRAME_DIR))[0]))
h0, w0 = first.shape[:2]
fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
out_orig = cv2.VideoWriter(os.path.join(OUTPUT_DIR, "orig.mp4"), fourcc, fps, (w0,h0))
out_bev  = cv2.VideoWriter(os.path.join(OUTPUT_DIR, "bird.mp4"), fourcc, fps, (W2+W1, H))

# debug: confirm writers opened
print("out_orig open?", out_orig.isOpened())
print("out_bev  open?", out_bev.isOpened())
print("Existing files before write:", sorted(os.listdir(OUTPUT_DIR)))

# ————————— PROCESS FRAMES —————————
for fn in sorted(os.listdir(FRAME_DIR)):
    frame = cv2.imread(os.path.join(FRAME_DIR, fn))
    if frame is None: continue

    # detect persons
    res = person_model(frame, conf=0.25, iou=0.4)
    pts2, boxes2 = [], []
    pts1, boxes1 = [], []

    for r in res:
        for b in r.boxes:
            x1,y1,x2,y2 = map(int, b.xyxy[0].tolist())
            if int(b.cls[0]) not in [2]: continue
            cx, cy = (x1+x2)//2, (y1+y2)//2
            if cv2.pointPolygonTest(poly2, (cx,cy), False)>=0:
                pts2.append((cx,cy)); boxes2.append((x1,y1,x2,y2))
            elif cv2.pointPolygonTest(poly1, (cx,cy), False)>=0:
                pts1.append((cx,cy)); boxes1.append((x1,y1,x2,y2))

    objs2 = tracker2.update(pts2)
    objs1 = tracker1.update(pts1)

    # draw on original
    for i,ctr in objs2.items():
        if tracker2.framesSeen[i]>=SKIP_DRAW_THRESHOLD and pts2:
            d = [np.hypot(ctr[0]-p[0],ctr[1]-p[1]) for p in pts2]
            if d:
                bb = boxes2[int(np.argmin(d))]
                cv2.rectangle(frame, bb[:2], bb[2:], (255,0,0), 2)
        if tracker2.framesSeen[i]>=MIN_FRAMES_THRESHOLD:
            counted2.add(i)

    for i,ctr in objs1.items():
        if tracker1.framesSeen[i]>=SKIP_DRAW_THRESHOLD and pts1:
            d = [np.hypot(ctr[0]-p[0],ctr[1]-p[1]) for p in pts1]
            if d:
                bb = boxes1[int(np.argmin(d))]
                cv2.rectangle(frame, bb[:2], bb[2:], (0,0,255), 2)
        if tracker1.framesSeen[i]>=MIN_FRAMES_THRESHOLD:
            counted1.add(i)

    cv2.putText(frame, f"bike-lane={len(counted2)}", (10,30),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)
    cv2.putText(frame, f"non-bike={len(counted1)}", (10,70),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),2)

    out_orig.write(frame)

    # bird’s-eye
    bird = canvas.copy()
    for (cx,cy) in pts2:
        p = cv2.perspectiveTransform(np.array([[[cx,cy]]],np.float32), H2)[0,0]
        cv2.circle(bird, (int(p[0]),int(p[1])), 6, (255,0,0), -1)
    for (cx,cy) in pts1:
        p = cv2.perspectiveTransform(np.array([[[cx,cy]]],np.float32), H1)[0,0]
        cv2.circle(bird, (int(p[0]),int(p[1])), 6, (0,0,255), -1)

    out_bev.write(bird)

out_orig.release()
out_bev.release()

print("Files after write:", sorted(os.listdir(OUTPUT_DIR)))


# Frames to video converter

In [ ]:
import cv2
import os

def create_video_from_frames(frames_folder, output_video_path, fps=30):
    """
    Combine frames from a folder into a video file.

    :param frames_folder: Path to the folder containing frames.
    :param output_video_path: Path to save the output video.
    :param fps: Frames per second for the video.
    """
    # Get the list of frame files
    frame_files = sorted([f for f in os.listdir(frames_folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])

    if not frame_files:
        print("No frames found in the folder.")
        return

    # Read the first frame to get dimensions
    first_frame_path = os.path.join(frames_folder, frame_files[0])
    first_frame = cv2.imread(first_frame_path)
    height, width, _ = first_frame.shape

    # Initialize the video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 videos
    video_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # Iterate over the frame files and write them to the video
    for frame_file in frame_files:
        frame_path = os.path.join(frames_folder, frame_file)
        frame = cv2.imread(frame_path)
        video_writer.write(frame)

    # Release the video writer
    video_writer.release()
    print(f"Video saved at: {output_video_path}")

# Inputs
frames_folder = '/content/drive/MyDrive/output_frames_folder'  # Folder containing processed frames
output_video_path = '/content/drive/MyDrive/output_video.mp4'  # Path for the output video
fps = 30  # Frames per second

# Create the video
create_video_from_frames(frames_folder, output_video_path, fps)
